In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%run ../../utils/reader_utils

In [0]:
%run ../../utils/writer_utils

In [0]:
%run ../../utils/transform_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants
SILVER_ORDERS_TABLE_CONF = {
    "table": "pyspark_scd2_orders",
    "schema": "silver",
    "timestamp_col": "_insert_update_ts"
}

DIM_CUSTOMERS_TABLE_CONF = {
    "table": "pyspark_dim_customer",
    "schema": "gold",
}

DIM_LOCATION_TABLE_CONF = {
    "table": "pyspark_dim_location",
    "schema": "gold",
}

DIM_DATE_TABLE_CONF = {
    "table": "pyspark_dim_date",
    "schema": "gold",
}

TARGET_TABLE_CONF = {
    "table": "pyspark_fact_order_fulfillment",
    "schema": "gold",
    "merge_keys": ["order_id"],
    "primary_key": "order_id"
}

STATUS_LIST = ["picked_up",
                "created",
                "out_for_delivery",
                "delivered",
                "failed_delivery",
                "in_transit",
                "returned",
                "confirmed"]

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
silver_orders_df = read_delta_table(SILVER_ORDERS_TABLE_CONF, START_DATE, END_DATE)

dim_customers_df = (spark.table(f"""{DIM_CUSTOMERS_TABLE_CONF.get("schema")}.
                        {DIM_CUSTOMERS_TABLE_CONF.get("table")}"""))

dim_location_df = (spark.table(f"""{DIM_LOCATION_TABLE_CONF.get("schema")}.
                        {DIM_LOCATION_TABLE_CONF.get("table")}"""))
                        
dim_date_df = (spark.table(f"""{DIM_DATE_TABLE_CONF.get("schema")}.
                        {DIM_DATE_TABLE_CONF.get("table")}"""))

In [0]:
orders_ids_updated = silver_orders_df.select("order_id")

In [0]:
updated_orders_df = (spark.table(f"""{SILVER_ORDERS_TABLE_CONF.get("schema")}.
                        {SILVER_ORDERS_TABLE_CONF.get("table")}""")
             .join(orders_ids_updated, "order_id", "inner")
             )

In [0]:
enriched_orders = updated_orders_df.select(*updated_orders_df.columns, 
                     *[F.when(F.col("status") == s, F.col("start_at")).otherwise(F.lit(None)).alias(f"{s}_ts") for s in STATUS_LIST]) 

In [0]:
window_spec = (Window.partitionBy("order_id")
     .orderBy("start_at")
     .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))

cols = [c for c in enriched_orders.columns if c not in ("order_id")]

pivoted_orders = (enriched_orders
    .select("order_id", 
            *[F.last(F.col(c), ignorenulls=True).over(window_spec).alias(c) for c in cols])
    .dropDuplicates(["order_id"])
    .withColumn("delivered_date", F.col("delivered_ts").cast(DateType())))

In [0]:
fact_order_fulfillment_df = (pivoted_orders.alias("main")
                             .join(dim_customers_df.alias("c"), 
                                   on=((F.col("main.customer_id") == F.col("c.customer_id"))
                                       & (F.col("main.start_at") >= F.col("c.start_at"))
                                       & ((F.col("main.start_at") < F.coalesce(F.col("c.end_at"), F.lit(today + timedelta(days=1)))))), 
                                   how="left")
                             .join(dim_location_df.alias("lo"), 
                                   on=(F.col("main.origin_city") == F.col("lo.city")), 
                                   how="left")
                             .join(dim_location_df.alias("ld"), 
                                   on=(F.col("main.destination_city") == F.col("ld.city")), 
                                   how="left")
                             .join(dim_date_df.alias("od"), 
                                   on=(F.col("main.order_date") == F.col("od.full_date")), 
                                   how="left")
                             .join(dim_date_df.alias("dd"), 
                                   on=(F.col("main.delivered_ts").cast(DateType()) == F.col("dd.full_date")), 
                                   how="left")
                             .select("order_id",
                                     "c.customer_sk",
                                     F.col("lo.location_sk").alias("orig_location_sk"),
                                     F.col("ld.location_sk").alias("dest_location_sk"),
                                     F.col("od.date_key").alias("order_date_key"),
                                     F.col("dd.date_key").alias("delivery_date_key"),
                                     "product_code",
                                    "product_name",
                                    "quantity",
                                    "total_amount",
                                    "payment_method",
                                    F.col("status").alias("current_status"),
                                    "confirmed_ts",
                                    "picked_up_ts",
                                    "in_transit_ts",
                                    "out_for_delivery_ts",
                                    "delivered_ts",
                                    "order_date",
                                     "delivery_date")
                             .withColumn("delivery_days", 
                                         F.datediff(F.col("delivery_date"), F.col("order_date")))
                             .withColumn("is_on_time", F.when(F.col("delivery_days") <= 30, F.lit(True)).otherwise(F.lit(False)))
                                     
)

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (fact_order_fulfillment_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY AUTO")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_ordr_customer_sk_fk FOREIGN KEY (customer_sk) 
              REFERENCES {DIM_CUSTOMERS_TABLE_CONF.get('schema')}.{DIM_CUSTOMERS_TABLE_CONF.get('table')}(customer_sk)""")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_ordr_orig_location_sk_fk FOREIGN KEY (orig_location_sk) 
              REFERENCES {DIM_LOCATION_TABLE_CONF.get('schema')}.{DIM_LOCATION_TABLE_CONF.get('table')}(location_sk)""")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_ordr_dest_location_sk_fk FOREIGN KEY (dest_location_sk) 
              REFERENCES {DIM_LOCATION_TABLE_CONF.get('schema')}.{DIM_LOCATION_TABLE_CONF.get('table')}(location_sk)""")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_ordr_order_date_key_fk FOREIGN KEY (order_date_key) 
              REFERENCES {DIM_DATE_TABLE_CONF.get('schema')}.{DIM_DATE_TABLE_CONF.get('table')}(date_key)""")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_ordr_delivery_date_key_fk FOREIGN KEY (delivery_date_key) 
              REFERENCES {DIM_DATE_TABLE_CONF.get('schema')}.{DIM_DATE_TABLE_CONF.get('table')}(date_key)""")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(fact_order_fulfillment_df, TARGET_TABLE_CONF)